In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 66.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=1453d852654595d2fc69c81086c692bd151b19808a47acd21344a49861829e8a
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.

# ── Overview of BB84 ────────────────────────────────────────────────────────
# 1. Alice generates random bits and random bases (Z or X).
# 2. Alice encodes each bit as a qubit in her chosen basis and sends it.
#      Z basis: bit 0 → |0⟩,  bit 1 → |1⟩
#      X basis: bit 0 → |+⟩,  bit 1 → |−⟩
# 3. Bob randomly chooses a measurement basis for each qubit and measures.
# 4. Alice and Bob publicly compare their bases (NOT the bits).
# 5. They keep only the bits where their bases agreed → shared secret key.
# 6. They compare a small subset of key bits to check for eavesdropping.
#    With no attacker, their keys must be identical.

backend = BasicSimulator()


In [3]:
# ── Quantum random number generation ────────────────────────────────────────
# The assignment requires using quantum measurement for randomness.
# We apply H to n qubits in state |0⟩ to get |+⟩ = 1/√2 (|0⟩ + |1⟩),
# then measure. Each qubit independently yields 0 or 1 with equal probability.

def quantum_random_bits(n):
    """Generate n random bits by measuring n qubits each prepared in |+⟩."""
    qc = QuantumCircuit(n, n)
    qc.h(range(n))            # H|0⟩ = |+⟩ = 1/√2 (|0⟩ + |1⟩) for every qubit
    qc.measure(range(n), range(n))
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]  # e.g. '01101'
    # Qiskit returns bits big-endian; reverse so index 0 = qubit 0
    return [int(b) for b in reversed(bitstring)]

# Test: generate 8 random bits
test_bits = quantum_random_bits(8)
print("Test random bits:", test_bits)

Test random bits: [0, 0, 0, 0, 0, 1, 0, 0]


In [4]:
# ── Alice: encoding ─────────────────────────────────────────────────────────
# Alice encodes a single bit in the chosen basis and returns the circuit.
# The circuit is sent (conceptually) to Bob over a quantum channel.
#
#   basis 0 (Z):  bit 0 → |0⟩  (do nothing)
#                 bit 1 → |1⟩  (apply X)
#   basis 1 (X):  bit 0 → |+⟩  (apply H)
#                 bit 1 → |−⟩  (apply X then H)

def alice_encode(bit, basis):
    """[ALICE] Encode `bit` in `basis` (0=Z, 1=X). Returns a 1-qubit circuit."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)       # flip |0⟩ to |1⟩
    if basis == 1:
        qc.h(0)       # rotate to X basis
    return qc

# ── Bob: measurement ─────────────────────────────────────────────────────────
# Bob applies H first if he chose the X basis (to rotate back to Z),
# then measures in the Z basis.

def bob_measure(qc, basis):
    """[BOB] Measure the qubit in `qc` using `basis` (0=Z, 1=X). Returns 0 or 1."""
    if basis == 1:
        qc.h(0)       # rotate from X basis back to Z before measuring
    qc.measure(0, 0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])

In [6]:
# ── Run the BB84 protocol (no attacker) ─────────────────────────────────────

n = 20              # number of qubits Alice sends
THRESHOLD = 0.10    # error rate above this → suspect an attacker

print("=" * 60)
print("BB84 Protocol — No Attacker")
print("=" * 60)

# ── Step 1: Alice generates random bits and bases ────────────────────────────
alice_bits  = quantum_random_bits(n)
alice_bases = quantum_random_bits(n)   # 0 = Z basis, 1 = X basis
print(f"\n[ALICE] Random bits:  {alice_bits}")
print(f"[ALICE] Random bases: {alice_bases}  (0=Z, 1=X)")

# ── Step 2: Alice encodes and sends; Bob measures with random bases ───────────
bob_bases   = quantum_random_bits(n)
bob_results = []

for i in range(n):
    qc  = alice_encode(alice_bits[i], alice_bases[i])   # [ALICE] encodes qubit i
    bit = bob_measure(qc, bob_bases[i])                 # [BOB]   measures qubit i
    bob_results.append(bit)

print(f"\n[BOB]   Random bases: {bob_bases}  (0=Z, 1=X)")
print(f"[BOB]   Measurements: {bob_results}")

# ── Step 3: Sifting — Alice and Bob compare bases publicly ───────────────────
# They discard any position where they used different bases.
sifted_alice = []
sifted_bob   = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:        # bases match → keep this bit
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"\n[PUBLIC] Bases compared. {len(sifted_alice)}/{n} positions matched.")
print(f"\n[ALICE] Sifted key: {sifted_alice}")
print(f"[BOB]   Sifted key: {sifted_bob}")

# ── Step 4: Error checking ───────────────────────────────────────────────────
# In a real protocol, they sacrifice a random subset of key bits for this check.
# Here we compare the entire sifted key for demonstration.
errors     = sum(a != b for a, b in zip(sifted_alice, sifted_bob))
error_rate = errors / len(sifted_alice) if sifted_alice else 0

print(f"\n[CHECK] Errors:     {errors}/{len(sifted_alice)}")
print(f"[CHECK] Error rate: {error_rate:.1%}  (threshold = {THRESHOLD:.0%})")

if error_rate > THRESHOLD:
    print("[CHECK]  ATTACK DETECTED — aborting key exchange.")
else:
    print("[CHECK]  No attack detected. Keys are identical — protocol succeeded!")
    print(f"\n[KEY]  Shared secret key: {sifted_alice}")

BB84 Protocol — No Attacker

[ALICE] Random bits:  [0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1]
[ALICE] Random bases: [1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1]  (0=Z, 1=X)

[BOB]   Random bases: [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1]  (0=Z, 1=X)
[BOB]   Measurements: [1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]

[PUBLIC] Bases compared. 14/20 positions matched.

[ALICE] Sifted key: [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]
[BOB]   Sifted key: [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]

[CHECK] Errors:     0/14
[CHECK] Error rate: 0.0%  (threshold = 10%)
[CHECK]  No attack detected. Keys are identical — protocol succeeded!

[KEY]  Shared secret key: [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]
